In [ ]:
import pandas as pd
import numpy as np

# Definir colunas desejadas (descarta Pais e outras não listadas)
colunas_desejadas = [
    'DataHora', 'Latitude', 'Longitude', 'Bioma',
    'Estado', 'Municipio', 'Satelite', 'FRP',
    'DiaSemChuva', 'Precipitacao', 'RiscoFogo'
]

caminhos = [
    '/content/drive/MyDrive/datasets_queimadas/bdqueimadas_2020-01-01_2021-01-01.csv',
    '/content/drive/MyDrive/datasets_queimadas/bdqueimadas_2021-01-01_2022-01-01.csv',
    '/content/drive/MyDrive/datasets_queimadas/bdqueimadas_2022-01-01_2023-01-01.csv',
    '/content/drive/MyDrive/datasets_queimadas/bdqueimadas_2023-01-01_2024-01-01.csv'
]

dfs_filtrados = []

print("--- Distribuição de Satélites no Total (Antes do Filtro) ---")
sat_counts = pd.Series(dtype='int64')
for c in caminhos:
    df_sat = pd.read_csv(c, usecols=lambda x: x.strip().lower() == 'satelite', encoding='utf-8-sig', dtype=str)
    sat_counts = sat_counts.add(df_sat.iloc[:, 0].str.strip().value_counts(), fill_value=0)
print(sat_counts.sort_values(ascending=False))
print("-" * 50)

for caminho in caminhos:
    # Lê apenas as colunas necessárias para economizar RAM
    df_temp = pd.read_csv(
        caminho,
        encoding='utf-8-sig',
        usecols=lambda c: c in colunas_desejadas,
        low_memory=False
    )

    # Padroniza textos (strip)
    for col in ['Bioma', 'Estado', 'Municipio', 'Satelite']:
        if col in df_temp.columns:
            df_temp[col] = df_temp[col].astype(str).str.strip()

    # Filtra Satélite de referência e Biomas logo na leitura
    df_temp = df_temp[df_temp['Satelite'] == 'AQUA_M-T']
    df_temp = df_temp[df_temp['Bioma'].isin(['Pantanal', 'Cerrado'])]

    dfs_filtrados.append(df_temp)

# Concatena a base já filtrada
df_combinado = pd.concat(dfs_filtrados, ignore_index=True)

# Exibe as informações do DataFrame concatenado
df_combinado.info()

--- Distribuição de Satélites no Total (Antes do Filtro) ---
Satelite
GOES-16      5618577.0
NOAA-20      5361723.0
NPP-375      3806384.0
NPP-375D     1699773.0
AQUA_M-T      797760.0
TERRA_M-T     343846.0
NOAA-19       337503.0
TERRA_M-M     236928.0
METOP-B       193812.0
NOAA-18D      130735.0
METOP-C       105067.0
AQUA_M-M      100675.0
MSG-03         69083.0
NOAA-19D       24606.0
NOAA-18         9206.0
NOAA-21          231.0
dtype: float64
--------------------------------------------------
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 272809 entries, 0 to 272808
Data columns (total 11 columns):
 #   Column        Non-Null Count   Dtype  
---  ------        --------------   -----  
 0   DataHora      272809 non-null  object 
 1   Satelite      272809 non-null  object 
 2   Estado        272809 non-null  object 
 3   Municipio     272809 non-null  object 
 4   Bioma         272809 non-null  object 
 5   DiaSemChuva   270734 non-null  float64
 6   Precipitacao  270734 non-nul

Limpeza de dados

In [ ]:
df_combinado = df_combinado.drop_duplicates()
print("Valores duplicados por coluna:",df_combinado.duplicated().sum())


Valores duplicados por coluna: 0


In [ ]:
# Tratar DataHora com formato exato e fuso em UTC
df_combinado['DataHora'] = df_combinado['DataHora'].astype(str).str.strip()
df_combinado['DataHora'] = pd.to_datetime(
    df_combinado['DataHora'],
    format='%Y/%m/%d %H:%M:%S',
    errors='coerce',
    utc=True
)

# Coluna 'Data' truncated para o dia (em UTC)
df_combinado['Data'] = df_combinado['DataHora'].dt.floor('D')

In [ ]:
# Tratar valores sentinela (-999 -> NaN) e ajustar dtypes
for col in ['FRP', 'DiaSemChuva', 'Precipitacao', 'RiscoFogo']:
    if col in df_combinado.columns:
        df_combinado[col] = pd.to_numeric(df_combinado[col], errors='coerce')
        df_combinado[col] = df_combinado[col].replace([-999, -999.0], np.nan).astype('float32')

# Coordenadas em float64 para não perder precisão na junção espacial com o INMET
df_combinado['Latitude'] = df_combinado['Latitude'].astype('float64')
df_combinado['Longitude'] = df_combinado['Longitude'].astype('float64')

In [ ]:
# Reordenar colunas
colunas_finais = [
    'DataHora', 'Data', 'Latitude', 'Longitude', 'Bioma',
    'Estado', 'Municipio', 'Satelite', 'FRP', 'DiaSemChuva',
    'Precipitacao', 'RiscoFogo'
]
df_combinado = df_combinado[colunas_finais]

# Salvar arquivo parquet final
caminho_saida = '/content/drive/MyDrive/datasets_queimadas/focos_pantanal_cerrado_2020_2023.parquet'
df_combinado.to_parquet(caminho_saida, index=False, compression='snappy')

In [ ]:
df_combinado.head()

,DataHora,Data,Latitude,Longitude,Bioma,Estado,Municipio,Satelite,FRP,DiaSemChuva,Precipitacao,RiscoFogo
0,2020-01-01 16:45:00+00:00,2020-01-01 00:00:00+00:00,-21.312,-48.225,Cerrado,SÃO PAULO,JABOTICABAL,AQUA_M-T,18.900000,3.0,7.0,0.2
1,2020-01-01 16:45:00+00:00,2020-01-01 00:00:00+00:00,-20.476,-52.129,Cerrado,MATO GROSSO DO SUL,TRÊS LAGOAS,AQUA_M-T,29.700001,3.0,0.1,0.2
2,2020-01-01 16:45:00+00:00,2020-01-01 00:00:00+00:00,-22.735,-49.486,Cerrado,SÃO PAULO,SANTA CRUZ DO RIO PARDO,AQUA_M-T,25.200001,8.0,2.2,0.9
3,2020-01-01 16:45:00+00:00,2020-01-01 00:00:00+00:00,-22.236,-47.194,Cerrado,SÃO PAULO,MOGI GUAÇU,AQUA_M-T,8.700000,4.0,5.7,0.4
4,2020-01-01 16:45:00+00:00,2020-01-01 00:00:00+00:00,-20.326,-48.235,Cerrado,SÃO PAULO,GUAÍRA,AQUA_M-T,17.799999,3.0,3.8,0.3


In [ ]:
print("\n1. Focos por Ano x Bioma:")
df_combinado['Ano'] = df_combinado['DataHora'].dt.year
print(pd.crosstab(df_combinado['Ano'], df_combinado['Bioma'], margins=True))
df_combinado.drop(columns=['Ano'], inplace=True)

print("\n2. % Nulos por coluna:")
print((df_combinado.isnull().sum() / len(df_combinado)) * 100)

print(f"\n3. Value Counts de 'Satelite' DEPOIS do filtro:")
print(df_combinado['Satelite'].value_counts(dropna=False))

print("\n4. Value Counts de 'Bioma':")
print(df_combinado['Bioma'].value_counts(dropna=False))

print(f"\n5. Shape final do DataFrame: {df_combinado.shape}")
print(f"6. Intervalo Temporal: Mínimo = {df_combinado['DataHora'].min()} | Máximo = {df_combinado['DataHora'].max()}")


1. Focos por Ano x Bioma:
Bioma   Cerrado  Pantanal     All
Ano                              
2020.0    63819     22116   85935
2021.0    62727      8258   70985
2022.0    56884      1637   58521
2023.0    50713      6580   57293
2024.0        4         1       5
All      234147     38592  272739

2. % Nulos por coluna:
DataHora        0.000733
Data            0.000733
Latitude        0.000000
Longitude       0.000000
Bioma           0.000000
Estado          0.000000
Municipio       0.000000
Satelite        0.000000
FRP             0.267287
DiaSemChuva     1.413429
Precipitacao    0.760795
RiscoFogo       1.340833
dtype: float64

3. Value Counts de 'Satelite' DEPOIS do filtro:
Satelite
AQUA_M-T    272741
Name: count, dtype: int64

4. Value Counts de 'Bioma':
Bioma
Cerrado     234149
Pantanal     38592
Name: count, dtype: int64

5. Shape final do DataFrame: (272741, 12)
6. Intervalo Temporal: Mínimo = 2020-01-01 16:45:00+00:00 | Máximo = 2024-01-01 18:00:00+00:00
